# Module 12 - 실습 3 솔루션: Send API Fan-out/Fan-in

In [ ]:
import sys
sys.path.insert(0, '../..')

import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, END
from langgraph.constants import Send
from common.fake_llm import FakeLLM

print("Send API 솔루션 준비 완료")

In [ ]:
class DocumentResult(TypedDict):
    doc_id: str
    summary: str
    word_count: int

class ParallelState(TypedDict):
    documents: list[dict]
    results: Annotated[list[DocumentResult], operator.add]  # 자동 합산!
    final_report: str | None

In [ ]:
def prepare_node(state: ParallelState) -> dict:
    docs = state.get("documents", [])
    print(f"[준비] {len(docs)}개 문서를 병렬 분석합니다.")
    return {}

def fan_out_documents(state: ParallelState) -> list[Send]:
    """각 문서에 대해 독립적인 분석 노드를 생성합니다 (Fan-out)."""
    documents = state.get("documents", [])
    
    if not documents:
        return [Send("generate_report", state)]
    
    sends = []
    for doc in documents:
        send_state = {
            **state,
            "_current_doc": doc,
            "results": [],  # 각 병렬 노드의 초기값
        }
        sends.append(Send("analyze_document", send_state))
    
    return sends

def analyze_document_node(state: ParallelState) -> dict:
    """개별 문서를 분석합니다 (병렬 실행됨)."""
    doc = state.get("_current_doc", {})
    doc_id = doc.get("id", "unknown")
    content = doc.get("content", "")
    
    print(f"  [분석] 문서 {doc_id} 분석 중...")
    
    word_count = len(content.split())
    summary = content[:50] + "..." if len(content) > 50 else content
    
    return {
        "results": [DocumentResult(doc_id=doc_id, summary=summary, word_count=word_count)],
    }

def generate_report_node(state: ParallelState) -> dict:
    """모든 병렬 분석 결과를 합쳐서 최종 리포트를 생성합니다 (Fan-in)."""
    results = state.get("results", [])
    
    report_lines = [
        f"멀티 문서 분석 리포트",
        f"{'='*40}",
        f"총 분석 문서: {len(results)}개",
        "",
    ]
    
    for result in results:
        report_lines.append(
            f"  [{result.get('doc_id', '?')}] "
            f"{result.get('summary', '')} "
            f"({result.get('word_count', 0)} words)"
        )
    
    total_words = sum(r.get("word_count", 0) for r in results)
    report_lines.append(f"\n총 단어 수: {total_words}")
    
    print(f"[리포트] 생성 완료")
    return {"final_report": "\n".join(report_lines)}

In [ ]:
def build_parallel_graph():
    graph = StateGraph(ParallelState)
    
    graph.add_node("prepare", prepare_node)
    graph.add_node("analyze_document", analyze_document_node)
    graph.add_node("generate_report", generate_report_node)
    
    graph.set_entry_point("prepare")
    
    # Fan-out: prepare 이후 각 문서별로 병렬 분석
    graph.add_conditional_edges(
        "prepare",
        fan_out_documents,
        ["analyze_document", "generate_report"],
    )
    
    # Fan-in: 모든 병렬 노드 완료 후 리포트 생성
    graph.add_edge("analyze_document", "generate_report")
    graph.add_edge("generate_report", END)
    
    return graph.compile()

app = build_parallel_graph()
print("그래프 구성 완료")

In [ ]:
result = app.invoke({
    "documents": [
        {"id": "DOC-001", "content": "Python은 간결하고 읽기 쉬운 프로그래밍 언어입니다."},
        {"id": "DOC-002", "content": "LangGraph는 LLM 에이전트 워크플로우를 구축하는 프레임워크입니다."},
        {"id": "DOC-003", "content": "체크포인팅은 장애 복구를 위한 핵심 기능입니다."},
    ],
    "results": [],
    "final_report": None,
})

print(result.get("final_report", "리포트가 없습니다"))

In [ ]:
assert result.get("final_report") is not None
assert result.get("results") is not None
assert len(result["results"]) == 3

doc_ids = {r.get("doc_id") for r in result["results"]}
assert "DOC-001" in doc_ids
assert "DOC-002" in doc_ids
assert "DOC-003" in doc_ids

print(f"✅ 실습 3 완료! {len(result['results'])}개 문서를 병렬로 분석했습니다.")